In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
PJME_data = pd.read_csv("PJME_phase2_preprocessed.csv",parse_dates=["Datetime"])

In [ ]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  ...  Rolling_Std_24  Rolling_Std_168
Datetime                            ...                                 
2002-01-08 01:00:00  29445.0     1  ...     4559.767709      3857.950565
2002-01-08 02:00:00  28670.0     2  ...     4425.965952      3861.770954
2002-01-08 03:00:00  28375.0     3  ...     4256.159403      3865.039821
2002-01-08 04:00:00  28542.0     4  ...     4064.104959      3864.924245
2002-01-08 05:00:00  29261.0     5  ...     3851.076461      3860.646270

[5 rows x 15 columns]


In [ ]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [ ]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [ ]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled = target_scaler.transform(y_val.values.reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1))

print("Scaling completed.")

Scaling completed.


In [ ]:
def create_sequences(X, y, input_steps=48, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(X[i-input_steps:i])
        y_seq.append(y[i:i+output_steps].flatten())

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101585, 48, 15)
y_train: (101585, 24)
X_val: (21713, 48, 15)
y_val: (21713, 24)
X_test: (21713, 48, 15)
y_test: (21713, 24)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
lstm_model_es = Sequential([
    LSTM(
        64,
        input_shape=(48, 15),
        return_sequences=False
    ),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_model_es.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_model_es.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
lstm_history_es = lstm_model_es.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1134 - mae: 0.2383 - val_loss: 0.0854 - val_mae: 0.2117
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0692 - mae: 0.1881 - val_loss: 0.0840 - val_mae: 0.2075
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0635 - mae: 0.1794 - val_loss: 0.0794 - val_mae: 0.2004
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0598 - mae: 0.1733 - val_loss: 0.0771 - val_mae: 0.1965
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 16ms/step - loss: 0.0567 - mae: 0.1686 - val_loss: 0.0802 - val_mae: 0.1960
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - loss: 0.0544 - mae: 0.1651 - val_loss: 0.0773 - val_mae: 0.1966
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0517 - mae: 0.1612 - val_loss: 0.0791 - val_mae: 0.1986
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 4.


In [ ]:
lstm_model_es_pred_scaled = lstm_model_es.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step


In [ ]:
lstm_pred = target_scaler.inverse_transform(
    lstm_model_es_pred_scaled.reshape(-1, 1)
).reshape(lstm_model_es_pred_scaled.shape)

lstm_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
lstm_mae = mean_absolute_error(lstm_actual.flatten(),lstm_pred.flatten())
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual.flatten(), lstm_pred.flatten()))
lstm_mape = np.mean(
    np.abs(
        (lstm_actual.flatten() - lstm_pred.flatten())
        / lstm_actual.flatten())) * 100
lstm_r2 = r2_score(lstm_actual.flatten(), lstm_pred.flatten())
lstm_bias = np.mean(lstm_pred.flatten() - lstm_actual.flatten())

print("LSTM + Earlystopping")
print("MAE :", lstm_mae)
print("RMSE:", lstm_rmse)
print("MAPE:", lstm_mape)
print("R²  :", lstm_r2)
print("Bias:", lstm_bias)

LSTM + Earlystopping
MAE : 1433.667664783884
RMSE: 2004.5797436739128
MAPE: 4.522895649228014
R²  : 0.9032423220484731
Bias: 102.46119586801278


In [ ]:
print("Epochs completed:", len(lstm_history_es.history["loss"]))

Epochs completed: 7


**Dropout**

In [ ]:
from tensorflow.keras.layers import LSTM, Dense, Dropout

lstm_dropout_model = Sequential([
    LSTM(
        64,
        input_shape=(48, 15),
        return_sequences=False
    ),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(24)
])

lstm_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_dropout_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_history_dropout = lstm_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - loss: 0.1824 - mae: 0.3208 - val_loss: 0.0979 - val_mae: 0.2344
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1231 - mae: 0.2654 - val_loss: 0.0945 - val_mae: 0.2323
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1138 - mae: 0.2547 - val_loss: 0.0862 - val_mae: 0.2181
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.1098 - mae: 0.2496 - val_loss: 0.0826 - val_mae: 0.2121
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.1070 - mae: 0.2463 - val_loss: 0.0814 - val_mae: 0.2111
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.1041 - mae: 0.2430 - val_loss: 0.0853 - val_mae: 0.2132
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - loss: 0.1025 - mae: 0.2409 - val_loss: 0.0814 - val_mae: 0.2107
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - loss: 0.1005 - mae: 0.2386 - val_loss: 0.0791 - val_mae: 0.2072
Epoch 9/15
1588/1588 ━━━━━

In [ ]:
dropout_pred_scaled = lstm_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


In [ ]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
mae = mean_absolute_error(dropout_actual.flatten(), dropout_pred.flatten())
rmse = np.sqrt(mean_squared_error(dropout_actual.flatten(), dropout_pred.flatten()))
mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten())) * 100
r2 = r2_score(dropout_actual.flatten(), dropout_pred.flatten())
bias = np.mean(dropout_pred.flatten() - dropout_actual.flatten())

print("== LSTM + DROPOUT ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== LSTM + DROPOUT ==
MAE : 1457.5810875911452
RMSE: 2022.0922473210635
MAPE: 4.648660141345585
R²  : 0.901544339372539
Bias: 245.40704667749569


**Batch Normalization**

In [ ]:
from tensorflow.keras.layers import  BatchNormalization

lstm_bn_model = Sequential([
    LSTM(
        64,
        input_shape=(48, 15)
    ),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(24)
])

lstm_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_bn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,712 (104.34 KB)

 Trainable params: 26,456 (103.34 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
history_bn = lstm_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.1690 - mae: 0.3018 - val_loss: 0.0981 - val_mae: 0.2299
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0953 - mae: 0.2319 - val_loss: 0.0880 - val_mae: 0.2149
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0863 - mae: 0.2197 - val_loss: 0.0804 - val_mae: 0.2055
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0818 - mae: 0.2137 - val_loss: 0.0812 - val_mae: 0.2041
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0780 - mae: 0.2085 - val_loss: 0.0771 - val_mae: 0.1998
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0755 - mae: 0.2058 - val_loss: 0.0778 - val_mae: 0.1975
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0707 - mae: 0.1992 - val_loss: 0.0952 - val_mae: 0.2237
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0691 - mae: 0.1976 - val_loss: 0.0802 - val_mae: 0.2005
Epoch 9/15
1588/1588 ━━━━━━━

In [ ]:
bn_pred_scaled = lstm_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [ ]:
lstm_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(bn_pred_scaled.shape)

lstm_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(lstm_bn_actual.flatten(), lstm_bn_pred.flatten())
rmse = np.sqrt(mean_squared_error(lstm_bn_actual.flatten(), lstm_bn_pred.flatten()))
mape = np.mean(np.abs((lstm_bn_actual.flatten() - lstm_bn_pred.flatten())/ lstm_bn_actual.flatten())) * 100
r2 = r2_score(lstm_bn_actual.flatten(), lstm_bn_pred.flatten())
bias = np.mean(lstm_bn_pred.flatten() - lstm_bn_actual.flatten())

print("== LSTM + Batch ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== LSTM + Batch ==
MAE : 1553.8807271944718
RMSE: 2239.3272076017715
MAPE: 4.8889870636177735
R²  : 0.8792536876928915
Bias: 279.824436144935


**RMSPROP**

In [ ]:
from tensorflow.keras.optimizers import RMSprop

lstm_rmsprop = Sequential([
    LSTM(
        64,
        input_shape=(48, 15),
        return_sequences=False
    ),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_rmsprop.compile(optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lstm_rmsprop.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_rmsprop = lstm_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1118 - mae: 0.2391 - val_loss: 0.0970 - val_mae: 0.2291
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.0720 - mae: 0.1923 - val_loss: 0.0948 - val_mae: 0.2183
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0655 - mae: 0.1820 - val_loss: 0.0822 - val_mae: 0.2084
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0619 - mae: 0.1760 - val_loss: 0.0816 - val_mae: 0.2115
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0590 - mae: 0.1716 - val_loss: 0.0789 - val_mae: 0.2010
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0565 - mae: 0.1678 - val_loss: 0.0805 - val_mae: 0.2050
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0542 - mae: 0.1644 - val_loss: 0.0810 - val_mae: 0.2039
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0521 - mae: 0.1615 - val_loss: 0.0773 - val_mae: 0.1975
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [ ]:
rmsprop_pred_scaled = lstm_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
rmsprop_mae = mean_absolute_error(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_rmse = np.sqrt(mean_squared_error(rmsprop_actual.flatten(), rmsprop_pred.flatten()))
rmsprop_mape = np.mean(np.abs((rmsprop_actual.flatten() - rmsprop_pred.flatten())/ rmsprop_actual.flatten()) * 100
rmsprop_r2 = r2_score(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_bias = np.mean(rmsprop_pred.flatten() - rmsprop_actual.flatten())

print("=== LSTM + RMSPROP ===")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

=== LSTM + RMSPROP ===
MAE : 1545.2959433009962
RMSE: 2209.6001114846167
MAPE: 4.885348178993356
R²  : 0.8824382266097829
Bias: 375.78220732779135


**SGD**

In [ ]:
from tensorflow.keras.optimizers import SGD

lstm_sgd = Sequential([
    LSTM(
        64,
        input_shape=(48, 15),
        return_sequences=False
    ),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_sgd.compile(optimizer=SGD(learning_rate=0.001,momentum=0.9),
    loss="mse",
    metrics=["mae"]
)

lstm_sgd.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,200 (102.34 KB)

 Trainable params: 26,200 (102.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_sgd = lstm_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.6037 - mae: 0.6045 - val_loss: 0.3514 - val_mae: 0.4744
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.2542 - mae: 0.4001 - val_loss: 0.1930 - val_mae: 0.3480
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1702 - mae: 0.3254 - val_loss: 0.1652 - val_mae: 0.3186
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.1459 - mae: 0.2986 - val_loss: 0.1483 - val_mae: 0.2979
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.1285 - mae: 0.2774 - val_loss: 0.1352 - val_mae: 0.2812
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.1162 - mae: 0.2613 - val_loss: 0.1267 - val_mae: 0.2707
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1078 - mae: 0.2497 - val_loss: 0.1209 - val_mae: 0.2619
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1020 - mae: 0.2412 - val_loss: 0.1172 - val_mae: 0.2567
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━

In [ ]:
sgd_pred_scaled = lstm_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
sgd_mae = mean_absolute_error(sgd_actual.flatten(), sgd_pred.flatten())
sgd_rmse = np.sqrt(mean_squared_error(sgd_actual.flatten(), sgd_pred.flatten()))
sgd_mape = np.mean(np.abs((sgd_actual.flatten() - sgd_pred.flatten()) / sgd_actual.flatten())) * 100
sgd_r2 = r2_score(sgd_actual.flatten(), sgd_pred.flatten())
sgd_bias = np.mean(sgd_pred.flatten() - sgd_actual.flatten())

print("=== LSTM + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== LSTM + SGD ===
MAE : 1626.9059845220413
RMSE: 2203.356937753004
MAPE: 5.173478138690356
R²  : 0.8831016241903846
Bias: 131.01977715593048


Learning Rate Scheduling

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

lstm_lr = Sequential([
    LSTM(
        64,
        input_shape=(48, 15)
    ),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
history_lr = lstm_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1126 - mae: 0.2395 - val_loss: 0.0931 - val_mae: 0.2220 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0707 - mae: 0.1909 - val_loss: 0.0860 - val_mae: 0.2166 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0650 - mae: 0.1816 - val_loss: 0.0970 - val_mae: 0.2282 - learning_rate: 0.0010
Epoch 4/15
1582/1588 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0616 - mae: 0.1766
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0613 - mae: 0.1758 - val_loss: 0.0900 - val_mae: 0.2188 - learning_rate: 0.0010
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0547 - mae: 0.1647 - val_loss: 0.0761 - val_mae: 0.1954 - learning_rate: 2.0000e-04
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0535 - mae: 0.1629 - val_loss: 0.0789 - val_mae: 0.1995 -

In [ ]:
lr_pred_scaled = lstm_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
lr_mae = mean_absolute_error(lr_actual.flatten(), lr_pred.flatten())
lr_rmse = np.sqrt(mean_squared_error(lr_actual.flatten(),lr_pred.flatten()))
lr_mape = np.mean(np.abs((lr_actual.flatten() - lr_pred.flatten()) / lr_actual.flatten())) * 100
lr_r2 = r2_score(lr_actual.flatten(), lr_pred.flatten())
lr_bias = np.mean(lr_pred.flatten() - lr_actual.flatten())

print("==== LSTM + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== LSTM + RMSPROP + LearningRate ====
MAE : 1430.0013529323771
RMSE: 2021.6438683401684
MAPE: 4.532692765106313
R²  : 0.9015879976719319
Bias: 278.14039346762905


**Layers**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_deep = Sequential([
    LSTM(64, return_sequences=True, input_shape=(48, 15)),
    LSTM(32, return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_8 (LSTM)                   │ (None, 48, 64)         │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,568 (142.84 KB)

 Trainable params: 36,568 (142.84 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_deep = lstm_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.1198 - mae: 0.2437 - val_loss: 0.0894 - val_mae: 0.2186
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0678 - mae: 0.1881 - val_loss: 0.0860 - val_mae: 0.2147
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0606 - mae: 0.1766 - val_loss: 0.0819 - val_mae: 0.2034
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0559 - mae: 0.1693 - val_loss: 0.0798 - val_mae: 0.2007
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0509 - mae: 0.1621 - val_loss: 0.0858 - val_mae: 0.2088
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0463 - mae: 0.1557 - val_loss: 0.0826 - val_mae: 0.2026
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 11ms/step - loss: 0.0415 - mae: 0.1487 - val_loss: 0.0891 - val_mae: 0.2115
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0375 - mae: 0.1424 - val_loss: 0.0919 - val_mae: 0.2140
Epoch 9/15
1588/1588 ━━━

In [ ]:
deep_pred_scaled = lstm_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
deep_mae = mean_absolute_error(deep_actual.flatten(), deep_pred.flatten())
deep_rmse = np.sqrt(mean_squared_error(deep_actual.flatten(), deep_pred.flatten()))
deep_mape = np.mean(np.abs((deep_actual.flatten() - deep_pred.flatten()) / deep_actual.flatten())) * 100
deep_r2 = r2_score(deep_actual.flatten(), deep_pred.flatten())
deep_bias = np.mean(deep_pred.flatten() - deep_actual.flatten())

print("==== LSTM + Layers ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

==== LSTM + Layers ==========
MAE : 1613.8322005319274
RMSE: 2322.8030119175005
MAPE: 5.063142075708516
R²  : 0.8700837359090798
Bias: 299.643297200266


**Neurons**

In [ ]:
lstm_128 = Sequential([
    LSTM(
        128,
        input_shape=(48, 15)
    ),
    Dense(
        64,
        activation="relu"
    ),
    Dense(24)
])

lstm_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_128.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 128)            │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,544 (326.34 KB)

 Trainable params: 83,544 (326.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_128 = lstm_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1008 - mae: 0.2241 - val_loss: 0.0899 - val_mae: 0.2154
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0644 - mae: 0.1810 - val_loss: 0.0864 - val_mae: 0.2087
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0578 - mae: 0.1706 - val_loss: 0.0795 - val_mae: 0.1996
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0526 - mae: 0.1631 - val_loss: 0.0816 - val_mae: 0.2015
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0464 - mae: 0.1541 - val_loss: 0.0864 - val_mae: 0.2030
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0408 - mae: 0.1455 - val_loss: 0.0835 - val_mae: 0.2027
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0356 - mae: 0.1375 - val_loss: 0.0878 - val_mae: 0.2035
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0308 - mae: 0.1291 - val_loss: 0.0913 - val_mae: 0.2073
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [ ]:
lstm128_pred_scaled = lstm_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lstm128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
lstm128_pred = target_scaler.inverse_transform(
    lstm128_pred_scaled.reshape(-1, 1)
).reshape(lstm128_pred_scaled.shape)

lstm128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
lstm128_mae = mean_absolute_error(lstm128_actual.flatten(), lstm128_pred.flatten())
lstm128_rmse = np.sqrt(mean_squared_error(lstm_actual.flatten(), lstm_pred.flatten()))
lstm128_mape = np.mean(np.abs((lstm128_actual.flatten() - lstm128_pred.flatten()) / lstm128_actual.flatten())) * 100
lstm128_r2 = r2_score(lstm128_actual.flatten(), lstm128_pred.flatten())
lstm128_bias = np.mean(lstm128_pred.flatten() - lstm128_actual.flatten())

print("=== LSTM 128 NEURONS ===")
print("MAE :", lstm128_mae)
print("RMSE:", lstm128_rmse)
print("MAPE:", lstm128_mape)
print("R²  :", lstm128_r2)
print("Bias:", lstm128_bias)

=== LSTM 128 NEURONS ===
MAE : 1601.2859191074658
RMSE: 2004.5797436739128
MAPE: 5.05863714722643
R²  : 0.8730547957973197
Bias: 285.11349263877355


32 **neurons**

In [ ]:
lstm_32 = Sequential([
    LSTM(
        32,
        input_shape=(48, 15)
    ),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_32.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_11 (LSTM)                  │ (None, 32)             │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,816 (38.34 KB)

 Trainable params: 9,816 (38.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_32 = lstm_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1372 - mae: 0.2610 - val_loss: 0.0937 - val_mae: 0.2239
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0736 - mae: 0.1957 - val_loss: 0.0859 - val_mae: 0.2105
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0683 - mae: 0.1867 - val_loss: 0.0838 - val_mae: 0.2085
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0654 - mae: 0.1819 - val_loss: 0.0822 - val_mae: 0.2056
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0631 - mae: 0.1781 - val_loss: 0.0776 - val_mae: 0.1998
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0612 - mae: 0.1753 - val_loss: 0.0786 - val_mae: 0.2010
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - loss: 0.0595 - mae: 0.1725 - val_loss: 0.0787 - val_mae: 0.1988
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0583 - mae: 0.1705 - val_loss: 0.0782 - val_mae: 0.1997
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [ ]:
lstm32_pred_scaled = lstm_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lstm32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
lstm32_pred = target_scaler.inverse_transform(
    lstm32_pred_scaled.reshape(-1, 1)
).reshape(lstm32_pred_scaled.shape)

lstm32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
lstm32_mae = mean_absolute_error(lstm32_actual.flatten(), lstm32_pred.flatten())
lstm32_rmse = np.sqrt(mean_squared_error(lstm32_actual.flatten(), lstm32_pred.flatten()))
lstm32_mape = np.mean( np.abs((lstm32_actual.flatten() - lstm32_pred.flatten()) / lstm32_actual.flatten())) * 100
lstm32_r2 = r2_score(lstm32_actual.flatten(), lstm32_pred.flatten())
lstm32_bias = np.mean(lstm32_pred.flatten() - lstm32_actual.flatten())

print("=== LSTM 32 NEURONS ===")
print("MAE :", lstm32_mae)
print("RMSE:", lstm32_rmse)
print("MAPE:", lstm32_mape)
print("R²  :", lstm32_r2)
print("Bias:", lstm32_bias)

=== LSTM 32 NEURONS ===
MAE : 1400.0472343464853
RMSE: 1995.0708248334797
MAPE: 4.411723882518581
R²  : 0.9041581037334473
Bias: 89.95396449458681


Batch Size(32)

In [ ]:
lstm_batch32 = Sequential([
    LSTM(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = lstm_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 25s 7ms/step - loss: 0.1016 - mae: 0.2253 - val_loss: 0.0835 - val_mae: 0.2100
Epoch 2/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0675 - mae: 0.1851 - val_loss: 0.0831 - val_mae: 0.2078
Epoch 3/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0618 - mae: 0.1758 - val_loss: 0.0815 - val_mae: 0.2019
Epoch 4/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0577 - mae: 0.1693 - val_loss: 0.0787 - val_mae: 0.1973
Epoch 5/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0543 - mae: 0.1642 - val_loss: 0.0805 - val_mae: 0.2015
Epoch 6/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0512 - mae: 0.1598 - val_loss: 0.0805 - val_mae: 0.1977
Epoch 7/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0478 - mae: 0.1551 - val_loss: 0.0816 - val_mae: 0.1991
Epoch 8/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - loss: 0.0445 - mae: 0.1506 - val_loss: 0.0880 - val_mae: 0.2059
Epoch 9/15
3175/3175 ━━━━━━━━━━━

In [ ]:
lstm_batch32_pred_scaled = lstm_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", lstm_batch32_pred_scaled.shape)

679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Prediction shape: (21713, 24)


In [ ]:
lstmbatch32_pred = target_scaler.inverse_transform(
    lstm_batch32_pred_scaled.reshape(-1, 1)
).reshape(lstm_batch32_pred_scaled.shape)

lstmbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
lstmb32_mae = mean_absolute_error(lstmbatch32_actual.flatten(), lstm32_pred.flatten())
lstmb32_rmse = np.sqrt(mean_squared_error(lstmbatch32_actual.flatten(), lstmbatch32_pred.flatten()))
lstmb32_mape = np.mean(
    np.abs(
        (lstmbatch32_actual.flatten() - lstmbatch32_pred.flatten())
        / lstmbatch32_actual.flatten())) * 100
lstmb32_r2 = r2_score(lstmbatch32_actual.flatten(), lstmbatch32_pred.flatten())
lstmb32_bias = np.mean(lstmbatch32_pred.flatten() - lstmbatch32_actual.flatten())

print("=== LSTM batch32 NEURONS ===")
print("MAE :", lstmb32_mae)
print("RMSE:", lstmb32_rmse)
print("MAPE:", lstmb32_mape)
print("R²  :", lstmb32_r2)
print("Bias:", lstmb32_bias)

=== LSTM batch32 NEURONS ===
MAE : 1400.0472343464853
RMSE: 2175.7268932440115
MAPE: 4.735649362757959
R²  : 0.8860150476039544
Bias: 207.61816333164103


BatchSize(128)

In [ ]:
lstm_batch128 = Sequential([
    LSTM(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = lstm_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.1408 - mae: 0.2636 - val_loss: 0.0906 - val_mae: 0.2197
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0715 - mae: 0.1926 - val_loss: 0.0831 - val_mae: 0.2062
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0655 - mae: 0.1830 - val_loss: 0.0807 - val_mae: 0.2065
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0618 - mae: 0.1767 - val_loss: 0.0802 - val_mae: 0.2042
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0594 - mae: 0.1728 - val_loss: 0.0808 - val_mae: 0.2013
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0575 - mae: 0.1698 - val_loss: 0.0829 - val_mae: 0.2056
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0555 - mae: 0.1667 - val_loss: 0.0766 - val_mae: 0.1957
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.0535 - mae: 0.1638 - val_loss: 0.0765 - val_mae: 0.1956
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - lo

In [ ]:
lstm_batch128_pred_scaled = lstm_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", lstm_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [ ]:
lstmbatch128_pred = target_scaler.inverse_transform(
    lstm_batch128_pred_scaled.reshape(-1, 1)
).reshape(lstm_batch128_pred_scaled.shape)

lstmbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
lstmb128_mae = mean_absolute_error(lstmbatch128_actual.flatten(), lstmbatch128_pred.flatten())
lstmb128_rmse = np.sqrt(mean_squared_error(lstmbatch128_actual.flatten(), lstmbatch128_pred.flatten()))
lstmb128_mape = np.mean(
    np.abs(
        (lstmbatch128_actual.flatten() - lstmbatch128_pred.flatten())
        / lstmbatch128_actual.flatten())) * 100
lstmb128_r2 = r2_score(lstmbatch128_actual.flatten(), lstmbatch128_pred.flatten())
lstmb128_bias = np.mean(lstmbatch128_pred.flatten() - lstmbatch128_actual.flatten())

print("=== LSTM batch128  ===")
print("MAE :", lstmb128_mae)
print("RMSE:", lstmb128_rmse)
print("MAPE:", lstmb128_mape)
print("R²  :", lstmb128_r2)
print("Bias:", lstmb128_bias)

=== LSTM batch128  ===
MAE : 1474.6325638243015
RMSE: 2107.1557296675155
MAPE: 4.64409547421797
R²  : 0.8930866275190343
Bias: 161.01738675613998


**HyperParameter Tuning**

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.0 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt

In [ ]:
def build_lstm(hp):

    model = Sequential()
    model.add(
        LSTM(
            units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]
            ),
            input_shape=(48, 15)
        )
    )
    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )
    model.add(Dense(24))
    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_lstm,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="lstm_48_to_24"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 03m 11s]
val_loss: 0.07670541107654572

Best val_loss So Far: 0.07636379450559616
Total elapsed time: 00h 19m 39s


In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 32
Dense Units: 128
Learning Rate: 0.001


In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,464 (52.59 KB)

 Trainable params: 13,464 (52.59 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0542 - mae: 0.1641 - val_loss: 0.0784 - val_mae: 0.2010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0530 - mae: 0.1621 - val_loss: 0.0793 - val_mae: 0.2000
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0521 - mae: 0.1608 - val_loss: 0.0759 - val_mae: 0.1963
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0512 - mae: 0.1594 - val_loss: 0.0814 - val_mae: 0.2040
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0501 - mae: 0.1580 - val_loss: 0.0834 - val_mae: 0.2046
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0493 - mae: 0.1569 - val_loss: 0.0804 - val_mae: 0.2006
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0485 - mae: 0.1559 - val_loss: 0.0800 - val_mae: 0.1992
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0474 - mae: 0.1544 - val_loss: 0.0816 - val_mae: 0.1997
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [ ]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [ ]:
best_mae = mean_absolute_error(best_actual.flatten(), best_pred.flatten())
best_rmse = np.sqrt(mean_squared_error(best_actual.flatten(), best_pred.flatten()))
best_mape = np.mean(np.abs((best_actual.flatten() - best_pred.flatten()) / best_actual.flatten())) * 100
best_r2 = r2_score(best_actual.flatten(), best_pred.flatten())
best_bias = np.mean(best_pred.flatten() - best_actual.flatten())

print("HYPERPARAMETER TUNED LSTM")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED LSTM
MAE : 1481.299239711035
RMSE: 2144.6536784990903
MAPE: 4.663846583152908
R²  : 0.8892476102732557
Bias: 332.19626600108876


**Additional Logs**

In [ ]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [ ]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144888, 21)


In [ ]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [ ]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101421, 21)
Validation: (21733, 21)
Test: (21734, 21)


In [ ]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [ ]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101350, 48, 21)
y_train: (101350, 24)
X_val: (21662, 48, 21)
y_val: (21662, 24)
X_test: (21663, 48, 21)
y_test: (21663, 24)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_lags = Sequential([
    LSTM(64, input_shape=(48, 21)),
    Dense(64, activation="relu"),
    Dense(24)
])

lstm_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_lags.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        22,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,736 (108.34 KB)

 Trainable params: 27,736 (108.34 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_lags = lstm_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1163 - mae: 0.2402 - val_loss: 0.0865 - val_mae: 0.2147
Epoch 2/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0686 - mae: 0.1872 - val_loss: 0.0770 - val_mae: 0.1997
Epoch 3/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0625 - mae: 0.1773 - val_loss: 0.0821 - val_mae: 0.2025
Epoch 4/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0585 - mae: 0.1709 - val_loss: 0.0773 - val_mae: 0.1941
Epoch 5/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0552 - mae: 0.1662 - val_loss: 0.0841 - val_mae: 0.2026
Epoch 6/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0519 - mae: 0.1619 - val_loss: 0.0787 - val_mae: 0.1993
Epoch 7/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0488 - mae: 0.1576 - val_loss: 0.0846 - val_mae: 0.2045
Epoch 8/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0454 - mae: 0.1531 - val_loss: 0.0762 - val_mae: 0.1939
Epoch 9/15
1584/1584 ━━━━━━━━━━━

In [ ]:
pred_scaled = lstm_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

339/339 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21663, 24)


In [ ]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [ ]:
mae = mean_absolute_error(actual.flatten(), pred.flatten())
rmse = np.sqrt(mean_squared_error(actual.flatten(), pred.flatten()))
mape = np.mean(np.abs((actual.flatten() - pred.flatten()) / actual.flatten())) * 100
r2 = r2_score(actual.flatten(), pred.flatten())
bias = np.mean(pred.flatten() - actual.flatten())

print("=== LSTM + ADDITIONAL LAGS ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== LSTM + ADDITIONAL LAGS ===
MAE : 1519.0271188321292
RMSE: 2243.3030576857145
MAPE: 4.756006436839991
R²  : 0.8788545826900508
Bias: 144.78844251827712


In [1]:
import pandas as pd
import os

lstm_48_24_results = [
    ["LSTM Baseline", 48, 24, 1480.8368071696918, 2166.632234680274, 4.637196960894945, 0.8869659827518424, 244.60307681727969],
    ["LSTM + EarlyStopping", 48, 24, 1433.667664783884, 2004.5797436739128, 4.522895649228014, 0.9032423220484731, 102.46119586801278],
    ["LSTM + Dropout", 48, 24, 1457.5810875911452, 2022.0922473210635, 4.648660141345585, 0.901544339372539, 245.40704667749569],
    ["LSTM + Batch Normalization", 48, 24, 1553.8807271944718, 2239.3272076017715, 4.8889870636177735, 0.8792536876928915, 279.824436144935],
    ["LSTM + RMSprop", 48, 24, 1545.2959433009962, 2209.6001114846167, 4.885348178993356, 0.8824382266097829, 375.78220732779135],
    ["LSTM + SGD", 48, 24, 1626.9059845220413, 2203.356937753004, 5.173478138690356, 0.8831016241903846, 131.01977715593048],
    ["LSTM + RMSprop + Learning Rate", 48, 24, 1430.0013529323771, 2021.6438683401684, 4.532692765106313, 0.9015879976719319, 278.14039346762905],
    ["LSTM + Additional Layers", 48, 24, 1613.8322005319274, 2322.8030119175005, 5.063142075708516, 0.8700837359090798, 299.643297200266],
    ["LSTM 128 Neurons", 48, 24, 1601.2859191074658, 2004.5797436739128, 5.05863714722643, 0.8730547957973197, 285.11349263877355],
    ["LSTM 32 Neurons", 48, 24, 1400.0472343464853, 1995.0708248334797, 4.411723882518581, 0.9041581037334473, 89.95396449458681],
    ["LSTM Batch Size 32", 48, 24, 1400.0472343464853, 2175.7268932440115, 4.735649362757959, 0.8860150476039544, 207.61816333164103],
    ["LSTM Batch Size 128", 48, 24, 1474.6325638243015, 2107.1557296675155, 4.64409547421797, 0.8930866275190343, 161.01738675613998],
    ["LSTM Hyperparameter Tuned", 48, 24, 1481.299239711035, 2144.6536784990903, 4.663846583152908, 0.8892476102732557, 332.19626600108876],
    ["LSTM + Additional Lags", 48, 24, 1519.0271188321292, 2243.3030576857145, 4.756006436839991, 0.8788545826900508, 144.78844251827712]
]

lstm_48_24_df = pd.DataFrame(
    lstm_48_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

# Display with exactly 6 decimal places
display(
    lstm_48_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

# Save inside existing comparison folder
os.makedirs("comparison", exist_ok=True)

lstm_48_24_df.to_csv(
    "comparison/LSTM_48h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/LSTM_48h_to_24h_results.csv")

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,LSTM Baseline,48,24,1480.836807,2166.632235,4.637197,0.886966,244.603077
1,LSTM + EarlyStopping,48,24,1433.667665,2004.579744,4.522896,0.903242,102.461196
2,LSTM + Dropout,48,24,1457.581088,2022.092247,4.648660,0.901544,245.407047
3,LSTM + Batch Normalization,48,24,1553.880727,2239.327208,4.888987,0.879254,279.824436
4,LSTM + RMSprop,48,24,1545.295943,2209.600111,4.885348,0.882438,375.782207
5,LSTM + SGD,48,24,1626.905985,2203.356938,5.173478,0.883102,131.019777
6,LSTM + RMSprop + Learning Rate,48,24,1430.001353,2021.643868,4.532693,0.901588,278.140393
7,LSTM + Additional Layers,48,24,1613.832201,2322.803012,5.063142,0.870084,299.643297
8,LSTM 128 Neurons,48,24,1601.285919,2004.579744,5.058637,0.873055,285.113493
9,LSTM 32 Neurons,48,24,1400.047234,1995.070825,4.411724,0.904158,89.953964


Saved: comparison/LSTM_48h_to_24h_results.csv


In [2]:
best_model = lstm_48_24_df.loc[lstm_48_24_df["MAE"].idxmin()]

display(best_model.to_frame().T.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)


,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
9,LSTM 32 Neurons,48,24,1400.047234,1995.070825,4.411724,0.904158,89.953964
